In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 04 · Model assessment (Task 4)

Score the models from Tasks 1–3. Do not treat VKM/`full` or `table_C_LEAKY_reference` as official results.

Protocol: 80/20 (`random_state=42`) for regression; 75/25 stratified for hotspots; 5-fold `KFold`; `GroupKFold` by Borough. Metrics: R², MAE, RMSE · ROC-AUC, precision, recall, F1.

In [2]:
!git clone --branch modelling-CTL-task-3 --single-branch https://github.com/stochasticquant/machine_learning_project.git

%cd /kaggle/working/machine_learning_project

!pwd
!ls

!pip install -r requirements_compatible.txt

fatal: destination path 'machine_learning_project' already exists and is not an empty directory.
/kaggle/working/machine_learning_project
/kaggle/working/machine_learning_project
data	   README.md  requirements_compatible.txt
notebooks  reports    requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 46.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 84.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.9/261.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 37.0 MB/s e

In [3]:
!find /kaggle/working/machine_learning_project/notebooks -maxdepth 2 -type f

/kaggle/working/machine_learning_project/notebooks/02_eda_and_profiling.ipynb
/kaggle/working/machine_learning_project/notebooks/modelling_fixed.ipynb
/kaggle/working/machine_learning_project/notebooks/.ipynb_checkpoints/modelling-checkpoint.ipynb
/kaggle/working/machine_learning_project/notebooks/.ipynb_checkpoints/modelling_fixed-checkpoint.ipynb


## 0 · Where the files are

A print out of all data locations.

In [5]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("Current working directory:", Path.cwd())

CANDIDATES = [
    # Running from project root
    Path("data/processed"),

    # Running from notebooks/
    Path("../data/processed"),

    # Kaggle cloned repo
    Path("/kaggle/working/machine_learning_project/data/processed"),

    # Kaggle attached datasets
    Path("/kaggle/input"),
]

PROC = None

for candidate in CANDIDATES:
    if not candidate.exists():
        continue

    if (candidate / "table_A_links.csv").exists():
        PROC = candidate.resolve()
        break

    matches = list(candidate.rglob("table_A_links.csv"))

    if matches:
        PROC = matches[0].parent.resolve()
        break

if PROC is None:
    raise FileNotFoundError(
        "Could not find table_A_links.csv. "
        f"Current working directory is {Path.cwd()}"
    )

print("processed dir:", PROC)
print("files:")

for path in sorted(PROC.glob("*")):
    if path.is_file():
        print(
            f"  {path.name:40s} "
            f"{path.stat().st_size / 1e6:6.2f} MB"
        )

Current working directory: /kaggle/working/machine_learning_project
processed dir: /kaggle/working/machine_learning_project/data/processed
files:
  grid_all_years.csv.gz                     26.67 MB
  link_features.csv                         18.87 MB
  link_targets.csv                           7.73 MB
  manifest.json                              0.00 MB
  table_A_links.csv                         32.36 MB
  table_B_grid_source_mix.csv                1.02 MB
  table_B_kmeans_matrix.npy                  0.44 MB
  table_B_with_clusters.csv                  1.08 MB
  table_C_LEAKY_reference.csv                0.63 MB
  table_C_grid_nonleaky.csv                  0.79 MB
  table_C_with_predictions.csv               0.91 MB


## 1 · Contract check

Confirm row counts, targets, and the realistic vs full feature split before fitting anything.

In [6]:
# Cell 2 · inspect the contract

manifest_path = PROC / "manifest.json"
with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

print("manifest keys:", list(manifest.keys()))
print("\ntables described:")
for name, spec in manifest.get("tables", {}).items():
    print(f"\n=== {name} ===")
    if isinstance(spec, dict):
        for k, v in spec.items():
            print(f"  {k}: {v}")
    else:
        print(" ", spec)

print("\nmodelling_protocol:")
print(json.dumps(manifest.get("modelling_protocol", {}), indent=2))

links = pd.read_csv(PROC / "table_A_links.csv", low_memory=False)
print("\ntable_A_links shape:", links.shape)
print("columns:")
print(list(links.columns))
print("\nhead:")
display(links.head(3))

# Target presence check
for col in ["nox", "pm10", "pm25", "co2", "nox_per_m", "Borough"]:
    print(f"  {col:12s} present={col in links.columns}")

manifest keys: ['project', 'source', 'base_year', 'excluded_years', 'environment', 'tables', 'modelling_protocol']

tables described:

=== table_A_links.csv ===
  algorithms: ['LinearRegression', 'RandomForestRegressor']
  unit: one major road link (OS TOID)
  rows: 79388
  targets: {'primary': 'nox', 'alternatives': ['pm10', 'pm25', 'co2', 'nox_per_m'], 'units': 'tonnes/year'}
  feature_sets: {'full': 43, 'realistic': 23}
  categorical: ['LAEI Zone', 'Borough', 'Road Classification']
  quality_flags: {'no_traffic_data': 688, 'bus_speed_missing': 26979}
  placeholder_handling: '-' in AADT/VKM means ZERO VEHICLES (verified: sum of classes == stated total) and is zero-filled. '-' in the Speed columns means MISSING and is left NaN for median imputation, with bus_speed_missing as an indicator.

=== table_B_grid_source_mix.csv ===
  algorithms: ['KMeans']
  unit: one 1 km grid cell
  rows: 3460
  features: ['share_Accidental Fires', 'share_Agriculture', 'share_Aviation', 'share_Biomass', 's

,TOID,LAEI Zone,Borough,Road Classification,zone_canon,is_motorway,no_traffic_data,bus_speed_missing,AADT Motorcycle,AADT Taxi,...,VKM 2019 - HGVs - Articulated - 5 Axles,VKM 2019 - HGVs - Articulated - 6 Axles,VKM 2019 - Buses,VKM 2019 - Coaches,VKM 2019 - Total,nox,pm10,pm25,co2,nox_per_m
0,osgb4000000027947700,Non-GLA,Non-GLA,A Road,Non GLA,0,0,1,112.0,15.0,...,2759.0,3659.0,0.0,327.0,231749.0,0.086293,0.004708,0.003169,41.891660,0.001541
1,osgb4000000027908760,Non-GLA,Non-GLA,A Road,Non GLA,0,0,1,68.0,10.0,...,1289.0,1718.0,0.0,1160.0,295268.0,0.103534,0.018034,0.008917,44.341598,0.000877
2,osgb4000000027987795,Non-GLA,Non-GLA,A Road,Non GLA,0,0,1,131.0,15.0,...,1042.0,1377.0,0.0,7746.0,158880.0,0.123533,0.012995,0.006589,47.647562,0.003743


  nox          present=True
  pm10         present=True
  pm25         present=True
  co2          present=True
  nox_per_m    present=True
  Borough      present=True


## 2 · Helpers

Return the R²/MAE/RMSE and the 23 / 43 feature counts.

In [7]:
# Cell 3 · helper module (same protocol as Stage 3)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SRC_CANDIDATES = [
    Path("/kaggle/working/machine_learning_project/src"),
    Path("../src"),
    Path("src"),
]
for src in SRC_CANDIDATES:
    if (src / "laei.py").exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))

try:
    import laei
    HAS_LAEI = True
except ImportError:
    laei = None
    HAS_LAEI = False

print("laei imported:", HAS_LAEI)


def regression_metrics(y_true, y_pred):
    if HAS_LAEI:
        return laei.regression_metrics(y_true, y_pred)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def classification_metrics(y_true, y_pred, y_prob=None):
    if HAS_LAEI and hasattr(laei, "classification_metrics"):
        if y_prob is None:
            return laei.classification_metrics(y_true, y_pred)
        return laei.classification_metrics(y_true, y_pred, y_prob)
    out = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }
    if y_prob is not None:
        out["ROC_AUC"] = float(roc_auc_score(y_true, y_prob))
        out["avg_precision"] = float(average_precision_score(y_true, y_prob))
    return out


def link_feature_sets(df):
    if HAS_LAEI:
        return laei.link_feature_sets(df)
    vkm = [c for c in df.columns if c.startswith("VKM")]
    aadt = [c for c in df.columns if c.startswith("AADT")]
    extra = [
        c for c in [
            "Speed (km/hr) - Except Buses",
            "Speed (km/hr) - Buses Only",
            "Link Length (m)",
        ]
        if c in df.columns
    ]
    return {
        "full": aadt + extra + vkm,
        "realistic": aadt + extra,
    }


def make_preprocessor(numeric, categorical):
    if HAS_LAEI:
        return laei.make_preprocessor(numeric, categorical)
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), categorical),
        ]
    )


CATEGORICAL = (
    laei.LINK_CATEGORICAL if HAS_LAEI
    else ["LAEI Zone", "Borough", "Road Classification"]
)
FEATURE_SETS = link_feature_sets(links)

print("categorical:", CATEGORICAL)
print({k: len(v) for k, v in FEATURE_SETS.items()})
print("expected from manifest:", manifest["tables"]["table_A_links.csv"]["feature_sets"])

laei imported: False
categorical: ['LAEI Zone', 'Borough', 'Road Classification']
{'full': 43, 'realistic': 23}
expected from manifest: {'full': 43, 'realistic': 23}


## 3 · Official link regression (realistic features)

Headline table for Task 4. Dummy median is the baseline. Linear and Ridge should be almost identical. Forest should beat both on the tail.

In [8]:
# Cell 4 · official realistic link regression (hold-out)

import time
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

numeric = FEATURE_SETS["realistic"]
y = links["nox"]
X = links[numeric + CATEGORICAL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def regressors():
    return {
        "DummyMedian": DummyRegressor(strategy="median"),
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0),
        "RandomForest": RandomForestRegressor(
            n_estimators=200, n_jobs=-1, random_state=42
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=300, random_state=42
        ),
    }

rows = []
for name, estimator in regressors().items():
    t0 = time.time()
    pipe = Pipeline([
        ("pre", make_preprocessor(numeric, CATEGORICAL)),
        ("model", estimator),
    ]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    met = regression_metrics(y_test, pred)
    rows.append({
        "feature_set": "realistic",
        "model": name,
        **met,
        "fit_s": round(time.time() - t0, 1),
    })
    print(
        f"{name:<22} "
        f"R2={met['R2']:.4f}  MAE={met['MAE']:.4f}  RMSE={met['RMSE']:.4f}"
    )

results_realistic = pd.DataFrame(rows)
display(results_realistic)

DummyMedian            R2=-0.0260  MAE=0.2215  RMSE=1.1011
LinearRegression       R2=0.7503  MAE=0.2060  RMSE=0.5432
Ridge                  R2=0.7503  MAE=0.2060  RMSE=0.5432
RandomForest           R2=0.9871  MAE=0.0191  RMSE=0.1233
HistGradientBoosting   R2=0.9114  MAE=0.0445  RMSE=0.3236


,feature_set,model,R2,MAE,RMSE,fit_s
0,realistic,DummyMedian,-0.026008,0.221470,1.101067,0.4
1,realistic,LinearRegression,0.750296,0.206018,0.543189,0.6
2,realistic,Ridge,0.750269,0.205992,0.543219,0.5
3,realistic,RandomForest,0.987132,0.019051,0.123307,179.3
4,realistic,HistGradientBoosting,0.911403,0.044525,0.323554,1.3


## 4 · Leaky contrast (`full` includes VKM)

Diagnostic only. Linear R² jumping toward 0.99 means the model was handed traffic × length, which is how LAEI builds the target.

In [9]:
# Cell 5 · leaky contrast (full = includes VKM). Not an official score.

numeric_full = FEATURE_SETS["full"]
X_full = links[numeric_full + CATEGORICAL]

Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

rows_full = []
for name, estimator in regressors().items():
    t0 = time.time()
    pipe = Pipeline([
        ("pre", make_preprocessor(numeric_full, CATEGORICAL)),
        ("model", estimator),
    ]).fit(Xtr_f, ytr_f)
    met = regression_metrics(yte_f, pipe.predict(Xte_f))
    rows_full.append({
        "feature_set": "full_LEAKY",
        "model": name,
        **met,
        "fit_s": round(time.time() - t0, 1),
    })
    print(
        f"{name:<22} "
        f"R2={met['R2']:.4f}  MAE={met['MAE']:.4f}  RMSE={met['RMSE']:.4f}"
    )

results_full = pd.DataFrame(rows_full)
compare = pd.concat([results_realistic, results_full], ignore_index=True)
display(
    compare.pivot(index="model", columns="feature_set", values=["R2", "MAE", "RMSE"])
    .round(4)
)

DummyMedian            R2=-0.0260  MAE=0.2215  RMSE=1.1011
LinearRegression       R2=0.9907  MAE=0.0334  RMSE=0.1048
Ridge                  R2=0.9907  MAE=0.0332  RMSE=0.1047
RandomForest           R2=0.9888  MAE=0.0150  RMSE=0.1152
HistGradientBoosting   R2=0.9502  MAE=0.0244  RMSE=0.2426


R2                  MAE                 RMSE  \
feature_set          full_LEAKY realistic full_LEAKY realistic full_LEAKY   
model                                                                       
DummyMedian             -0.0260   -0.0260     0.2215    0.2215     1.1011   
HistGradientBoosting     0.9502    0.9114     0.0244    0.0445     0.2426   
LinearRegression         0.9907    0.7503     0.0334    0.2060     0.1048   
RandomForest             0.9888    0.9871     0.0150    0.0191     0.1152   
Ridge                    0.9907    0.7503     0.0332    0.2060     0.1047   

                                
feature_set          realistic  
model                           
DummyMedian             1.1011  
HistGradientBoosting    0.3236  
LinearRegression        0.5432  
RandomForest            0.1233  
Ridge                   0.5432

## 5 · Five-fold CV

Checks that the hold-out ranking is not a lucky split.

In [12]:
# Cell 6 · 5-fold CV on the official realistic set

from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)
X_real = links[FEATURE_SETS["realistic"] + CATEGORICAL]

cv_rows = []
for name, estimator in regressors().items():
    if name == "DummyMedian":
        continue
    pipe = Pipeline([
        ("pre", make_preprocessor(FEATURE_SETS["realistic"], CATEGORICAL)),
        ("model", estimator),
    ])
    scores = cross_val_score(pipe, X_real, y, cv=cv, scoring="r2", n_jobs=1)
    print(f"{name:<22} {scores.mean():.4f} ± {scores.std():.4f}   folds={np.round(scores, 4)}")
    cv_rows.append({
        "model": name,
        "cv_R2_mean": float(scores.mean()),
        "cv_R2_std": float(scores.std()),
    })

results_cv = pd.DataFrame(cv_rows)
display(results_cv.round(4))

LinearRegression       0.7561 ± 0.0124   folds=[0.7503 0.7583 0.7382 0.7762 0.7577]
Ridge                  0.7561 ± 0.0124   folds=[0.7503 0.7583 0.7382 0.7762 0.7577]
RandomForest           0.9870 ± 0.0013   folds=[0.9874 0.9861 0.9881 0.9886 0.985 ]
HistGradientBoosting   0.9030 ± 0.0162   folds=[0.907  0.8711 0.9153 0.9083 0.9131]


,model,cv_R2_mean,cv_R2_std
0,LinearRegression,0.7561,0.0124
1,Ridge,0.7561,0.0124
2,RandomForest,0.9870,0.0013
3,HistGradientBoosting,0.9030,0.0162


## 6 · Borough-grouped CV

Asks whether Random Forest still works when a whole borough is unseen. 

In [13]:
# Cell 7 · GroupKFold by Borough (Random Forest, realistic)

from sklearn.model_selection import GroupKFold

pipe_rf = Pipeline([
    ("pre", make_preprocessor(FEATURE_SETS["realistic"], CATEGORICAL)),
    ("model", RandomForestRegressor(
        n_estimators=200, n_jobs=-1, random_state=42
    )),
])

gkf = GroupKFold(n_splits=5)
g_scores = cross_val_score(
    pipe_rf,
    X_real,
    y,
    cv=gkf,
    groups=links["Borough"],
    scoring="r2",
    n_jobs=1,
)
print("GroupKFold R2 by borough:", np.round(g_scores, 4))
print(f"mean {g_scores.mean():.4f} ± {g_scores.std():.4f}")

GroupKFold R2 by borough: [0.9798 0.988  0.9703 0.9809 0.9442]
mean 0.9727 ± 0.0153


## 7 · Top 10% of emitting links

MAE and RMSE on the tail. 

In [14]:
# Cell 8 · overall vs top-decile error (same split as Cell 4)

numeric = FEATURE_SETS["realistic"]
X_train, X_test, y_train, y_test = train_test_split(
    links[numeric + CATEGORICAL], links["nox"],
    test_size=0.2, random_state=42
)

fitted = {}
for name, estimator in {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=200, n_jobs=-1, random_state=42
    ),
}.items():
    fitted[name] = Pipeline([
        ("pre", make_preprocessor(numeric, CATEGORICAL)),
        ("model", estimator),
    ]).fit(X_train, y_train)

thr = y_test.quantile(0.90)
high = y_test >= thr
print(f"test n={len(y_test):,}  top-decile n={int(high.sum()):,}  threshold={thr:.4f} t/yr")

rows_slice = []
for name, pipe in fitted.items():
    pred = pipe.predict(X_test)
    for label, mask in [("all_test", np.ones(len(y_test), dtype=bool)), ("top_10pct", high.to_numpy())]:
        met = regression_metrics(y_test[mask], pred[mask])
        rows_slice.append({"model": name, "slice": label, **met})
        print(f"{name:<18} {label:<10} R2={met['R2']:.4f}  MAE={met['MAE']:.4f}  RMSE={met['RMSE']:.4f}")

results_slice = pd.DataFrame(rows_slice)
display(results_slice.round(4))

test n=15,878  top-decile n=1,588  threshold=0.4269 t/yr
LinearRegression   all_test   R2=0.7503  MAE=0.2060  RMSE=0.5432
LinearRegression   top_10pct  R2=0.7722  MAE=0.6318  RMSE=1.4799
RandomForest       all_test   R2=0.9871  MAE=0.0191  RMSE=0.1233
RandomForest       top_10pct  R2=0.9844  MAE=0.1265  RMSE=0.3878


,model,slice,R2,MAE,RMSE
0,LinearRegression,all_test,0.7503,0.2060,0.5432
1,LinearRegression,top_10pct,0.7722,0.6318,1.4799
2,RandomForest,all_test,0.9871,0.0191,0.1233
3,RandomForest,top_10pct,0.9844,0.1265,0.3878


## 8 · K-Means on source shares

Type the cell, do not predict tonnes. k=4 is for interpretability. Coordinates stay out of the fit.

In [15]:
# Cell 9 · Table B clustering assessment

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

table_b = pd.read_csv(PROC / "table_B_grid_source_mix.csv")
X_b = np.load(PROC / "table_B_kmeans_matrix.npy")
print("B rows:", table_b.shape, "matrix:", X_b.shape)

scan = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_b)
    sil = silhouette_score(X_b, km.labels_)
    scan.append({
        "k": k,
        "inertia": float(km.inertia_),
        "silhouette": float(sil),
    })
    print(f"k={k:<2} inertia={km.inertia_:8.0f}  silhouette={sil:.3f}")

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_b)
labels = kmeans.labels_
print("\nk=4 sizes:\n", pd.Series(labels).value_counts().sort_index().to_string())
print(
    "k=4 silhouette={:.3f}  CH={:.1f}  DB={:.3f}".format(
        silhouette_score(X_b, labels),
        calinski_harabasz_score(X_b, labels),
        davies_bouldin_score(X_b, labels),
    )
)

share_cols = [c for c in table_b.columns if c.startswith("share_")]
table_b = table_b.copy()
table_b["cluster"] = labels
profile = table_b.groupby("cluster")[share_cols].mean() * 100
print("\ncluster profiles (% of NOx-by-source mix):")
display(profile.round(1).T.sort_values(0, ascending=False).head(8))

B rows: (3460, 25) matrix: (3460, 16)
k=2  inertia=   33808  silhouette=0.250
k=3  inertia=   30482  silhouette=0.272
k=4  inertia=   27331  silhouette=0.272
k=5  inertia=   24545  silhouette=0.271
k=6  inertia=   21343  silhouette=0.284
k=7  inertia=   19395  silhouette=0.255
k=8  inertia=   16416  silhouette=0.278
k=9  inertia=   13940  silhouette=0.299
k=10 inertia=   11973  silhouette=0.321

k=4 sizes:
 0      69
1    1253
2    2071
3      67
k=4 silhouette=0.272  CH=452.2  DB=1.335

cluster profiles (% of NOx-by-source mix):


cluster,0,1,2,3
share_Aviation,63.9,0.3,0.2,1.6
share_Road Transport,18.0,26.9,68.0,8.8
share_Heat and Power Generation,13.4,56.6,25.0,12.7
share_Waste,1.4,0.1,0.2,0.4
share_Construction,1.1,1.8,3.0,0.9
share_Industrial Processes,1.0,3.6,1.1,3.6
share_River,0.9,0.5,0.2,71.5
share_Agriculture,0.2,6.9,0.9,0.5


## 9 · Hotspot classification (non-leaky grid)

10% class. Dummy accuracy will look strong and F1 will be zero.

In [16]:
# Cell 10 · Table C hotspot classification (non-leaky features)

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

table_c = pd.read_csv(PROC / "table_C_grid_nonleaky.csv")
feat_c = manifest["tables"]["table_C_grid_nonleaky.csv"]["features"]
print("C shape:", table_c.shape)
print("hotspot rate:", table_c["nox_hotspot"].mean())

Xc = table_c[feat_c]
yc = table_c["nox_hotspot"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.25, stratify=yc, random_state=42
)

dummy = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
print("dummy most_frequent:", classification_metrics(yc_test, dummy.predict(Xc_test)))

clf_models = {
    "LogisticRegression": Pipeline([
        ("sc", StandardScaler()),
        ("m", LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=42
        )),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=400, n_jobs=-1, class_weight="balanced", random_state=42
    ),
}

proba = {}
clf_rows = []
for name, est in clf_models.items():
    est.fit(Xc_train, yc_train)
    pred = est.predict(Xc_test)
    pr = est.predict_proba(Xc_test)[:, 1]
    proba[name] = pr
    met = classification_metrics(yc_test, pred, pr)
    clf_rows.append({"model": name, "threshold": 0.5, **met})
    print(name, met)

results_clf = pd.DataFrame(clf_rows)
display(results_clf.round(4))

C shape: (3460, 25)
hotspot rate: 0.1
dummy most_frequent: {'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'accuracy': 0.900578034682081}
LogisticRegression {'precision': 0.5985401459854015, 'recall': 0.9534883720930233, 'F1': 0.7354260089686099, 'accuracy': 0.9317919075144508, 'ROC_AUC': 0.9815655133295519, 'avg_precision': 0.9169786176497642}
RandomForest {'precision': 0.9655172413793104, 'recall': 0.6511627906976745, 'F1': 0.7777777777777778, 'accuracy': 0.9630057803468208, 'ROC_AUC': 0.9888198943188943, 'avg_precision': 0.9160069897005777}


,model,threshold,precision,recall,F1,accuracy,ROC_AUC,avg_precision
0,LogisticRegression,0.5,0.5985,0.9535,0.7354,0.9318,0.9816,0.917
1,RandomForest,0.5,0.9655,0.6512,0.7778,0.9630,0.9888,0.916


## 10 · Cost-tuned threshold

Miss costs 5× a false alarm. After retuning, logistic regression and the forest should look similar.

In [17]:
# Cell 11 · cost-sensitive threshold (miss=5, false alarm=1)

from sklearn.metrics import confusion_matrix

COST_MISS, COST_FA = 5.0, 1.0

best_thresholds = {}
for name, pr in proba.items():
    best = None
    for t in np.arange(0.05, 0.96, 0.05):
        tn, fp, fn, tp = confusion_matrix(
            yc_test, (pr >= t).astype(int), labels=[0, 1]
        ).ravel()
        cost = COST_MISS * fn + COST_FA * fp
        if best is None or cost < best[0]:
            best = (cost, t, fn, fp)
    best_thresholds[name] = best[1]
    print(
        f"{name}: best t={best[1]:.2f}, cost={best[0]:.0f} "
        f"(missed {best[2]}, false alarms {best[3]})"
    )

rows_t = []
for name, pr in proba.items():
    t = best_thresholds[name]
    pred = (pr >= t).astype(int)
    met = classification_metrics(yc_test, pred, pr)
    rows_t.append({"model": name, "threshold": t, **met})
    print(name, f"at {t:.2f}:", met)

results_clf_cost = pd.DataFrame(rows_t)
display(results_clf_cost.round(4))

LogisticRegression: best t=0.65, cost=64 (missed 5, false alarms 39)
RandomForest: best t=0.25, cost=63 (missed 5, false alarms 38)
LogisticRegression at 0.65: {'precision': 0.675, 'recall': 0.9418604651162791, 'F1': 0.7864077669902912, 'accuracy': 0.9491329479768786, 'ROC_AUC': 0.9815655133295519, 'avg_precision': 0.9169786176497642}
RandomForest at 0.25: {'precision': 0.680672268907563, 'recall': 0.9418604651162791, 'F1': 0.7902439024390244, 'accuracy': 0.9502890173410404, 'ROC_AUC': 0.9888198943188943, 'avg_precision': 0.9160069897005777}


,model,threshold,precision,recall,F1,accuracy,ROC_AUC,avg_precision
0,LogisticRegression,0.65,0.6750,0.9419,0.7864,0.9491,0.9816,0.917
1,RandomForest,0.25,0.6807,0.9419,0.7902,0.9503,0.9888,0.916


## 11 · Intensity (`nox_per_m`)

Same features, different question. Linear should improve a lot once length is in the denominator of the target.

In [19]:
# Cell 12 · intensity target (nox per metre)

mask = links["nox_per_m"].notna()
numeric = FEATURE_SETS["realistic"]
X_int = links.loc[mask, numeric + CATEGORICAL]
y_int = links.loc[mask, "nox_per_m"]

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    X_int, y_int, test_size=0.2, random_state=42
)

rows_int = []
for name, estimator in regressors().items():
    if name == "DummyMedian":
        continue
    t0 = time.time()
    pipe = Pipeline([
        ("pre", make_preprocessor(numeric, CATEGORICAL)),
        ("model", estimator),
    ]).fit(Xtr_i, ytr_i)
    met = regression_metrics(yte_i, pipe.predict(Xte_i))
    rows_int.append({"target": "nox_per_m", "model": name, **met,
                     "fit_s": round(time.time() - t0, 1)})
    print(f"{name:<22} R2={met['R2']:.4f}  MAE={met['MAE']:.4f}  RMSE={met['RMSE']:.4f}")

results_intensity = pd.DataFrame(rows_int)
display(results_intensity)

LinearRegression       R2=0.9124  MAE=0.0003  RMSE=0.0006
Ridge                  R2=0.9125  MAE=0.0003  RMSE=0.0006
RandomForest           R2=0.9899  MAE=0.0001  RMSE=0.0002
HistGradientBoosting   R2=0.9692  MAE=0.0002  RMSE=0.0004


,target,model,R2,MAE,RMSE,fit_s
0,nox_per_m,LinearRegression,0.912409,0.000332,0.000628,0.5
1,nox_per_m,Ridge,0.912458,0.000332,0.000628,0.4
2,nox_per_m,RandomForest,0.989914,0.000088,0.000213,137.2
3,nox_per_m,HistGradientBoosting,0.969206,0.000221,0.000372,1.4


## 12 · Final Assesment


* Official link model: Random Forest on realistic features (no VKM) predicts link NOx at R² 0.987, MAE 0.019 t/year; 5-fold CV is 0.987 ± 0.001 and borough-grouped CV is still 0.973.

* Linear models are a different tool: without VKM they only reach R² 0.75 and barely beat a median dummy on MAE, because emissions are traffic × length, not a sum. Add VKM and they jump to R² 0.99 — that is leakage, not skill.

* The tail is the real test: on the busiest 10% of links, linear MAE rises to 0.63 and RMSE to 1.48; the forest stays much tighter (0.13 / 0.39). MAE and RMSE tell that story; R² alone does not.

* Intensity vs totals: on nox_per_m, linear improves to R² 0.91. If the question is “how dirty is this metre of road?”, a linear explainer is usable; if the question is “how many tonnes?”, use the forest.

* K-Means (k=4, silhouette 0.27) types cells (aviation / heat-and-power / roads / river), it does not predict tonnes. Hotspot classifiers both reach AUC ~0.98; after a cost-tuned threshold they miss 5 cells each and are interchangeable. Dummy accuracy of 90% is meaningless.
